# Atividade Prática 3 — 03 · Etapa 2: warm-start e busca (roda e salva)

> **Este notebook só RODA e SALVA as buscas** em `results/curvas/<método>.npz` (curvas, configurações avaliadas e desenhos
> iniciais de cada alvo e semente). Tudo tem cache: o que já está em `results/curvas/` é lido em vez de recalculado (use
> `FORCE_SEARCH = True` para refazer). As análises, figuras e testes estão no **`04_Analises.ipynb`**, que só lê esses arquivos.
> Rodar do zero leva ~2 h em 11 núcleos (17 métodos SMBO × 120 alvos × 5 sementes × até 100 ajustes de GP).

**Protocolo.** Para cada um dos 120 alvos d_t (leave-one-dataset-out), **T = 100** avaliações por busca, **n_init = 20** pontos
iniciais (os mesmos do protocolo do paper do D2V) e **5 sementes**. Avaliar uma configuração = ler a célula de d_t na matriz de
perdas. O meta-conhecimento usado para montar o warm-start vem só dos outros 119 datasets.

1. **Padrão** — a configuração padrão (fixa, sem busca).
2. **Random Search** — permutação aleatória das K configurações, sem reposição.
3. **SMBO frio** — 20 configurações aleatórias, depois GP + EI.
4. **WS-⟨rep⟩ (k)** — os 20 pontos iniciais vêm dos *k* datasets mais próximos de d_t no espaço ⟨rep⟩ ∈ {PyMFE, MF1, MF2, D2V,
   Our-Trained}, em rodízio (a melhor configuração de cada vizinho, depois a 2ª melhor de cada um, ...); k ∈ {1, 3, 5}. Com k = 1
   são as 20 melhores do vizinho mais próximo (protocolo do paper do D2V); com k ≥ 20 seria exatamente Feurer et al. (2014).
5. **WS independente** — as 20 configurações de melhor rank médio nos 119 datasets de treino, sem olhar para d_t.

**Controle extra (Análise 2): WS-Random (k)** — o mesmo método 4, mas com uma representação aleatória (32 dimensões ~ U(−1, 1),
semente 42): os k "vizinhos" são datasets arbitrários. Separa o efeito de *usar as melhores configurações de k datasets* do efeito
de *escolhê-los por similaridade*.

Os métodos 3–5 usam o **mesmo SMBO**: substituto **GP com kernel Matérn 3/2 e ARD** (hiperparâmetros reotimizados a cada
iteração) sobre a codificação one-hot/min-max das configurações, e aquisição **Expected Improvement**, maximizada sobre as
configurações ainda não avaliadas. Empates (comuns: há muitas configurações com a mesma acurácia de validação) são desfeitos por
sorteio com a semente — tanto na ordenação das configurações de um vizinho quanto no argmax do EI. A busca para quando acha o
ótimo do grid (a curva de melhor-até-agora não muda mais; isso só economiza tempo).

In [1]:
import os, sys, warnings
from pathlib import Path
warnings.filterwarnings("ignore")

ROOT = Path.cwd()
if not (ROOT / "src" / "config.py").exists():
    ROOT = Path(r"D:\Projetos\DisciplinaMtL\entrega3")
os.chdir(ROOT); sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from src import config as cfg, search, analysis as an
from src.metatask_data import load_meta_task

an.setup_matplotlib()
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30)
cfg.make_dirs()

def save_fig(fig, name):
    for ext in ("png", "pdf"):
        fig.savefig(cfg.FIGURES_DIR / f"{name}.{ext}", bbox_inches="tight", facecolor="white")

def save_table(df, name, index=True):
    df.to_csv(cfg.TABLES_DIR / f"{name}.csv", index=index)

FORCE_SEARCH = False     # True: refaz as buscas mesmo com cache em results/curvas/
N_JOBS = -2              # processos do joblib (-2 = todos os núcleos menos 1)

meta = load_meta_task()
REPS = cfg.REP_ALL + cfg.REP_CONTROL               # + controle: representação aleatória (vizinhos arbitrários)
ctx = search.build_context(meta, reps=REPS)
KEYS = search.all_method_keys(REPS)
print(f"{len(ctx['names'])} alvos | K={meta.n_cfg} | T={ctx['T']} | n_init={ctx['n_init']} | sementes={ctx['n_seeds']} | "
      f"{len(KEYS)} métodos")
print("Configuração padrão:", meta.configs.iloc[ctx['default_idx']].to_dict())

120 alvos | K=3456 | T=100 | n_init=20 | sementes=5 | 22 métodos
Configuração padrão: {'ACTIVATION': 'relu', 'ARCHITECTURE': 'SQU', 'DROPOUT': 0.0, 'LAYERS': 1, 'NEURONS': 32, 'NORMALIZATION': 0, 'OPTIMZERS': 'adam'}


In [2]:
RES = {}
for key in KEYS:
    cached = None if FORCE_SEARCH else search.load_method(key, ctx)
    RES[key] = cached if cached is not None else search.run_method(key, ctx, n_jobs=N_JOBS, force=FORCE_SEARCH)
    print(f"{'cache ' if cached is not None else 'rodado'} | {search.method_label(key):24s} | "
          f"tempo de parede {RES[key]['info']['wall_s']:7.0f} s | serial {RES[key]['info']['serial_s']:7.0f} s")

cache  | Padrão                   | tempo de parede       0 s | serial       0 s
cache  | Random Search            | tempo de parede       0 s | serial       0 s
cache  | SMBO frio                | tempo de parede     485 s | serial    5184 s
cache  | WS independente          | tempo de parede     489 s | serial    5198 s
cache  | WS-PyMFE (k=1)           | tempo de parede     515 s | serial    5512 s
cache  | WS-PyMFE (k=3)           | tempo de parede     477 s | serial    5055 s
cache  | WS-PyMFE (k=5)           | tempo de parede     422 s | serial    4365 s
cache  | WS-MF1 (k=1)             | tempo de parede     519 s | serial    5582 s
cache  | WS-MF1 (k=3)             | tempo de parede     452 s | serial    4761 s
cache  | WS-MF1 (k=5)             | tempo de parede     424 s | serial    4501 s
cache  | WS-MF2 (k=1)             | tempo de parede     481 s | serial    5013 s
cache  | WS-MF2 (k=3)             | tempo de parede     462 s | serial    4819 s
cache  | WS-MF2 (k=5)       

cache  | WS-Our-Trained (k=5)     | tempo de parede     337 s | serial    3616 s
cache  | WS-Random (k=1)          | tempo de parede     498 s | serial    5290 s
cache  | WS-Random (k=3)          | tempo de parede     439 s | serial    4721 s
cache  | WS-Random (k=5)          | tempo de parede     471 s | serial    5032 s


## Conferência rápida

Perda normalizada média (sobre alvos e sementes) logo após os pontos iniciais, no orçamento intermediário e em T. Os testes
estatísticos e as figuras estão no `04_Analises.ipynb`.

In [3]:
quick = pd.DataFrame({search.method_label(k): {f"t={t}": r["curves"][:, :, t - 1].mean() for t in cfg.CHECKPOINTS}
                      for k, r in RES.items()}).T
quick["AUC"] = [r["curves"].mean() for r in RES.values()]
quick["tempo serial (s)"] = [r["info"]["serial_s"] for r in RES.values()]
quick["tempo de parede (s)"] = [r["info"]["wall_s"] for r in RES.values()]
save_table(quick, "buscas_conferencia")
display(quick.style.format("{:.4f}", subset=[c for c in quick.columns if not c.startswith("tempo")])
        .format("{:.0f}", subset=["tempo serial (s)", "tempo de parede (s)"]))
print(f"Custo total das buscas simuladas: {quick['tempo serial (s)'].sum() / 3600:.1f} h de CPU "
      f"({quick['tempo de parede (s)'].sum() / 3600:.1f} h de parede).")
files = sorted(cfg.CURVES_DIR.glob("*.npz"))
print(f"{len(files)} arquivos em {cfg.CURVES_DIR.relative_to(ROOT)}/")

,t=20,t=50,t=100,AUC,tempo serial (s),tempo de parede (s)
Padrão,0.1700,0.1700,0.1700,0.1700,0,0
Random Search,0.0834,0.0606,0.0467,0.0722,0,0
SMBO frio,0.0842,0.0520,0.0375,0.0652,5184,485
WS independente,0.0755,0.0526,0.0377,0.0604,5198,489
WS-PyMFE (k=1),0.0733,0.0453,0.0341,0.0549,5512,515
WS-PyMFE (k=3),0.0635,0.0452,0.0338,0.0531,5055,477
WS-PyMFE (k=5),0.0645,0.0438,0.0326,0.0525,4365,422
WS-MF1 (k=1),0.0771,0.0491,0.0363,0.0593,5582,519
WS-MF1 (k=3),0.0670,0.0476,0.0365,0.0554,4761,452
WS-MF1 (k=5),0.0601,0.0431,0.0335,0.0515,4501,424


Custo total das buscas simuladas: 26.7 h de CPU (2.5 h de parede).
22 arquivos em results\curvas/
